# Anticipación del Conflicto Legal en Siniestros Laborales (NYSWCB)
### Notebook reproducible — Tesis de Maestría en Ciencia de Datos · ITBA 2026
**Autor:** Julián Tapia · **Director:** Ariel Aizemberg

Este notebook reproduce el pipeline central de la tesis:
**carga de datos → target de litigio evitable → validación temporal → CatBoost v3 → métricas (test 2022) → interacción AWW×C-3 → SHAP → impacto económico + Monte Carlo → auditoría de equidad → verificación exacta con el modelo de la tesis.**

> Las secciones 3 a 8 **reentrenan** el modelo final con los mismos hiperparámetros y el mismo preprocesamiento que `src/08_retrain_v3_gpu.py`. Un reentrenamiento en otro hardware no es bit a bit idéntico, por lo que sus cifras coinciden con las de la tesis de forma aproximada. La **sección 9** carga el modelo exacto de la tesis y reproduce las cifras publicadas sin reentrenar; se puede correr sola después de las secciones 0 a 2.
>
> El benchmark de 9 modelos y la optimización con Optuna (60 trials, ~5,7 h en GPU) están en `src/`. Las referencias a secciones corresponden a la versión final de la tesis (`Tesis_Completa_v42_ITBA_rec`).

## 0 · Dependencias

In [ ]:
!pip -q install catboost scikit-learn pyarrow pandas numpy scipy > /dev/null
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
print("Listo")

## 1 · Cargar el dataset analítico

El dataset (`dataset_tesis_clean.parquet`, 1.592.919 reclamos × 25 columnas, ~25 MB) viene **incluido en el repositorio** y se descarga automáticamente desde un commit fijo. No hay que subir nada ni montar Drive.

> Es el subconjunto filtrado del [dataset público del NYSWCB](https://data.ny.gov/Government-Finance/Assembled-Workers-Compensation-Claims-Beginning-20/jshw-gkgu), guardado en Parquet (comprimido) para que entre en GitHub.
>
> Los scripts de la tesis leen este mismo dataset desde CSV. Al pasar por CSV, los códigos WCIO y de industria se leen como números (`9.0`) y no como texto (`09`). Para que las variables categóricas tengan exactamente los mismos valores que en el entrenamiento de la tesis, la celda reproduce ese paso en memoria.

In [ ]:
import pandas as pd, urllib.request, os, io

COMMIT = "830023d93155de94f32ac15cf7e0794f4573cb3b"   # commit fijo del repositorio
RAW    = f"https://github.com/Jtapia2211/TesisITBA/raw/{COMMIT}/"

def descargar(rel):
    """Usa la copia local si se clonó el repo; si no, descarga del commit fijo."""
    local = os.path.join("..", rel)
    if os.path.exists(local):
        return local
    dst = os.path.basename(rel)
    if not os.path.exists(dst):
        print("Descargando", rel, "...")
        urllib.request.urlretrieve(RAW + rel, dst)
    return dst

df = pd.read_parquet(descargar("data/dataset_tesis_clean.parquet"))

# Mismos tipos que ve pandas.read_csv en los scripts de src/ (Parquet -> CSV -> read_csv)
buf = io.StringIO(); df.to_csv(buf, index=False); buf.seek(0)
df = pd.read_csv(buf, low_memory=False); del buf

print("Filas:", f"{len(df):,}", "| Columnas:", df.shape[1])
df.head(3)

## 2 · Target de litigio evitable y validación temporal (§3.3.2 · §4.6.1)

- **Target refinado = 1** si el caso judicializó **y** la resolución es no permanente (`NON-COMP`, `MED ONLY`, `TEMPORARY`). Los casos de incapacidad permanente / fallecimiento (litigio estructuralmente inevitable) pasan a 0.
- `claim_injury_type_REF` se usa **solo** para construir el target y luego se **excluye** de las features (anti-leakage, §3.4).
- **Split temporal estricto:** entrenamiento 2017–2020 · validación 2021 · prueba 2022.

In [ ]:
POS = {'2. NON-COMP','3. MED ONLY','4. TEMPORARY'}
df['target_ref'] = ((df['target']==1) & (df['claim_injury_type_REF'].isin(POS))).astype(int)

# Mismas variables y mismo orden que src/08_retrain_v3_gpu.py
num_features = ['days_to_assembly','days_C2_to_accident','days_C3_to_accident','age_at_injury','aww',
                'has_C2','has_C3','has_ANCR_early','accident_year','accident_month','accident_dow']
cat_features = ['gender','accident_type','occupational_disease','county_of_injury','medical_fee_region',
                'wcio_cause_code','wcio_nature_code','wcio_body_code','carrier_type','district_name',
                'industry_code','industry_desc']
features = num_features + cat_features
print(f"{len(features)} features ({len(cat_features)} categóricas, {len(num_features)} numéricas)")

tr = df[df.accident_year<=2020].copy()
va = df[df.accident_year==2021].copy()
te = df[df.accident_year==2022].copy()
for name,d in [('train',tr),('val',va),('test',te)]:
    print(f"{name:5s} n={len(d):>9,}  prevalencia target_ref={d.target_ref.mean()*100:5.2f}%")

In [ ]:
# Preprocesamiento idéntico al de src/08_retrain_v3_gpu.py:
# categóricas -> texto, faltantes = "UNKNOWN"; numéricas -> faltantes = -1
def prep(d):
    d = d.copy()
    for c in cat_features: d[c] = d[c].astype(object).where(d[c].notna(), "UNKNOWN").astype(str)
    for c in num_features: d[c] = pd.to_numeric(d[c], errors='coerce').fillna(-1)
    return d
tr,va,te = prep(tr),prep(va),prep(te)

Xtr,ytr = tr[features], tr['target_ref']
Xva,yva = va[features], va['target_ref']
Xte,yte = te[features], te['target_ref']

# Peso de la clase positiva: negativos/positivos del dataset completo (como en la tesis)
SPW = 5.631
print("scale_pos_weight =", SPW, "| calculado sobre el dataset completo:",
      round((df.target_ref==0).sum()/(df.target_ref==1).sum(), 3))

## 3 · Modelo final CatBoost v3 (§4.10.2 · §4.14)

Hiperparámetros óptimos hallados por la búsqueda bayesiana (Optuna, §4.11–4.16):
`depth=10 · learning_rate=0.061 · l2_leaf_reg=7.62 · bagging_temperature=0.792 · border_count=82`.

> En CPU el entrenamiento completo tarda del orden de 90 minutos. Si solo querés comprobar las cifras de la tesis, podés saltar a la sección 9.

In [ ]:
from catboost import CatBoostClassifier, Pool
try:
    from catboost.utils import get_gpu_device_count
    TASK = 'GPU' if get_gpu_device_count() > 0 else 'CPU'
except Exception:
    TASK = 'CPU'
print("Entrenando en:", TASK)

pool_tr = Pool(Xtr, ytr, cat_features=cat_features)
pool_va = Pool(Xva, yva, cat_features=cat_features)
pool_te = Pool(Xte, yte, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=1511, depth=10, learning_rate=0.061, l2_leaf_reg=7.62,
    bagging_temperature=0.792, border_count=82, loss_function='Logloss',
    eval_metric='PRAUC', scale_pos_weight=SPW, random_seed=42,
    task_type=TASK, early_stopping_rounds=100, verbose=200)
model.fit(pool_tr, eval_set=pool_va)
print("Iteraciones efectivas:", model.tree_count_)

## 4 · Evaluación en prueba 2022 (§4.10.2 · §5.2)

Valores de la tesis: **AUC-ROC 0,8833 · PR-AUC 0,5981 · Recall 67,8 % · Precisión 51,9 %** con τ = 0,7084 (TP 27.903 · FP 25.868 · FN 13.245 · TN 193.140).
Al reentrenar aquí, las cifras deberían coincidir de forma aproximada (diferencias en el tercer o cuarto decimal); la reproducción exacta está en la sección 9.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from scipy.stats import ks_2samp

# Umbral de la tesis: máximo F1 sobre la grilla de best_f1_threshold (150 valores en [0,05; 0,95]).
# La tesis lo informa redondeado (0,7084); en validación 2021 el máximo F1 está en 0,6903 (§5.3).
GRID    = np.linspace(0.05, 0.95, 150)
TAU     = float(GRID[np.argmin(np.abs(GRID - 0.7084))])
TAU_VAL = float(GRID[np.argmin(np.abs(GRID - 0.6903))])

def evaluar(y, p, tau):
    pred = (p >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return tp, fp, fn, tn, pred

p_te = model.predict_proba(pool_te)[:,1]
auc = roc_auc_score(yte, p_te); prauc = average_precision_score(yte, p_te)
ks = ks_2samp(p_te[yte==1], p_te[yte==0]).statistic
tp, fp, fn, tn, pred = evaluar(yte, p_te, TAU)
recall = tp/(tp+fn); prec = tp/(tp+fp)
print(f"AUC-ROC : {auc:.4f}")
print(f"PR-AUC  : {prauc:.4f}")
print(f"KS      : {ks:.4f}")
print(f"\nMatriz @ τ={TAU:.4f}:  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"Recall  : {recall*100:.1f}%   Precisión: {prec*100:.1f}%")

## 5 · Interacción Salario × Formulario C-3 (§4.21 · Tabla 27)

Tasa de litigio evitable real (prueba 2022) por las cuatro combinaciones AWW(=0/>0) × C-3.
Valores de la tesis: **3,2 % / 26,7 % / 26,2 % / 49,0 %**.

In [ ]:
te2 = te.copy(); te2['AWW>0'] = (te2['aww']>0).astype(int)
seg = (te2.groupby(['AWW>0','has_C3'])
          .agg(n=('target_ref','size'), tasa=('target_ref','mean')))
seg['tasa %'] = (seg['tasa']*100).round(1); seg['share %']=(seg['n']/len(te2)*100).round(1)
print(seg[['n','tasa %','share %']])
hi = te2[(te2['AWW>0']==1)&(te2.has_C3==1)].target_ref.sum()
print(f"\nAWW>0 & C-3 concentra el {hi/te2.target_ref.sum()*100:.0f}% de los litigios reales")

## 6 · Interpretabilidad SHAP (§4.17–4.22)

La tesis calcula SHAP sobre el modelo optimizado del target original (`src/09_cap7_shap.py`). Esta celda aplica el mismo método (TreeSHAP exacto, muestra estratificada de 4.000 positivos y 4.000 negativos) al modelo recién entrenado, como ilustración: el orden de las variables debería ser similar, no idéntico.

In [ ]:
# Muestra estratificada 50/50: 4.000 positivos + 4.000 negativos
pos = te[te.target_ref==1].sample(min(4000,(te.target_ref==1).sum()), random_state=0)
neg = te[te.target_ref==0].sample(4000, random_state=0)
samp = pd.concat([pos, neg])

expl = model.get_feature_importance(
    Pool(samp[features], samp['target_ref'], cat_features=cat_features), type='ShapValues')
sv = expl[:, :-1]
order = np.argsort(np.abs(sv).mean(0))[::-1][:10]
for i in order:
    print(f"{features[i]:24s}  mean|SHAP| = {np.abs(sv[:,i]).mean():.4f}")

## 7 · Impacto económico (§5.2 · §5.4)

Parámetros base: `C_j = $13.000` (litigio evitable) · `C_p = $3.400` (mediación privada) · `C_fp = $500` (falsa alarma).
Valores de la tesis: cota superior con efectividad plena **$255 M, ROI 2,4×**. Monte Carlo con los supuestos de la Tabla 35 (tasa de resolución de la mediación entre 40 % y 60 %): mediana **$72 M [IC 90 %: $23 M – $125 M]**, ROI mediano 0,7×; con mediación pública, mediana $134 M.

La celda usa la matriz de confusión obtenida en la sección 4; con el modelo reentrenado las cifras son aproximadas. `src/10_montecarlo_cap8.py` las reproduce con la matriz exacta de la tesis.

In [ ]:
Cj,Cp,Cfp = 13000,3400,500
ahorro = tp*(Cj-Cp) - fp*Cfp
inv = tp*Cp + fp*Cfp
print(f"Cota superior (p_éxito=100%): ahorro ${ahorro/1e6:.1f}M | inversión ${inv/1e6:.1f}M | ROI {ahorro/inv:.2f}x")
print(f"Reducción vs sin modelo: {ahorro/((tp+fn)*Cj)*100:.1f}%")

# Monte Carlo (Tabla 35): mismas distribuciones, semilla y orden que src/10_montecarlo_cap8.py
rng = np.random.default_rng(42); N = 50_000
cj   = rng.triangular(9000, 13000, 17000, N)     # costo de litigio
cfp  = rng.triangular(350, 500, 650, N)          # falsa alarma
cp   = rng.triangular(2100, 3400, 4800, N)       # mediación privada
cpub = rng.triangular(850, 1200, 1600, N)        # mediación pública
pex  = rng.triangular(0.40, 0.50, 0.60, N)       # tasa de resolución de la mediación
mc     = (tp*(pex*cj - cp)   - fp*cfp)/1e6
mc_pub = (tp*(pex*cj - cpub) - fp*cfp)/1e6
inv_mc = (tp*cp + fp*cfp)/1e6
print(f"\nMonte Carlo (referencia): P50 ${np.percentile(mc,50):.0f}M  [P5 ${np.percentile(mc,5):.0f}M – P95 ${np.percentile(mc,95):.0f}M]")
print(f"P(ahorro>0) = {(mc>0).mean()*100:.1f}%   ROI mediano = {np.median(mc/inv_mc):.2f}x")
print(f"Mediación pública: P50 ${np.percentile(mc_pub,50):.0f}M")

## 8 · Auditoría de equidad — regla de los cuatro quintos (§6.3.1)

Equal Opportunity (igualdad de sensibilidad) por género y por quintil de salario (AWW). Como referencia analítica, se considera disparidad un cociente entre grupos inferior a 0,80.
Valores de la tesis: cociente de sensibilidad F/M = 1,003; por quintil de AWW, la sensibilidad baja de 0,861 a 0,639 (cociente 0,742).

In [ ]:
te3 = te.copy(); te3['pred']=pred; te3['y']=yte.values
def tpr(d):
    pos=d[d.y==1]; return pos.pred.mean() if len(pos) else np.nan
print("== Por género ==")
g = pd.DataFrame({k: {'n': len(d), 'TPR': tpr(d)} for k, d in te3.groupby('gender')}).T
g = g[g.n>500]; print(g.round(3))
print(f"Ratio TPR(F)/TPR(M) = {g.loc['F','TPR']/g.loc['M','TPR']:.3f}")

print("\n== Por quintil de AWW (sobre AWW>0) ==")
pos = te3[te3.aww>0].copy()
pos['Q'] = pd.qcut(pos.aww,5,labels=[1,2,3,4,5])
q = pd.DataFrame({k: {'n': len(d), 'TPR': tpr(d)} for k, d in pos.groupby('Q', observed=True)}).T
q['ratio_vs_Q1'] = (q.TPR/q.TPR.iloc[0]).round(3)
print(q.round(3))
r = q.TPR.iloc[-1]/q.TPR.iloc[0]
print("\nQ5/Q1 =", round(r,3), "→", "sin disparidad" if r>=0.8 else "disparidad (ver calibración por quintil, §6.3.1)")

## 9 · Verificación exacta con el modelo de la tesis (sin reentrenar)

Descarga el modelo final exacto (`models/catboost_v3_full.cbm`, ~81 MB) del mismo commit y predice sobre la prueba 2022. Aquí las cifras deben coincidir **exactamente** con las de la tesis. Requiere haber corrido las secciones 0 a 2 y la celda de preprocesamiento. Es lo mismo que hace `verificar_metricas.py` en el repositorio.

In [ ]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

GRID    = np.linspace(0.05, 0.95, 150)
TAU     = float(GRID[np.argmin(np.abs(GRID - 0.7084))])
TAU_VAL = float(GRID[np.argmin(np.abs(GRID - 0.6903))])

tesis = CatBoostClassifier(); tesis.load_model(descargar("models/catboost_v3_full.cbm"))
p_ex = tesis.predict_proba(Pool(Xte, cat_features=cat_features))[:,1]

print(f"AUC-ROC : {roc_auc_score(yte, p_ex):.4f}   (tesis: 0,8833)")
print(f"PR-AUC  : {average_precision_score(yte, p_ex):.4f}   (tesis: 0,5981)")
for nombre, tau in (("τ = 0,7084 (máximo F1 en prueba 2022)", TAU), ("τ = 0,6903 (máximo F1 en validación 2021)", TAU_VAL)):
    tn_, fp_, fn_, tp_ = confusion_matrix(yte, (p_ex >= tau).astype(int)).ravel()
    print(f"\n{nombre}")
    print(f"  TP={tp_:,}  FP={fp_:,}  FN={fn_:,}  TN={tn_:,}")
    print(f"  Recall {tp_/(tp_+fn_)*100:.1f}%   Precisión {tp_/(tp_+fp_)*100:.1f}%")
print("\nTesis, τ = 0,7084: TP=27,903  FP=25,868  FN=13,245  TN=193,140 · Recall 67,8% · Precisión 51,9%")

---
### Notas
- Las cifras económicas son proyecciones bajo supuestos explícitos, no validadas en producción.
- La calibración diferenciada del umbral por quintil (`src/14_fairness_calibration.py`, §6.3.1) eleva la sensibilidad del quintil superior; aquí se muestra el estado previo a la calibración.
- Pipeline completo, benchmark de 9 modelos y Optuna: ver `src/` en el repositorio.